# Metadata Agent — AgentCore Batch Evaluations

This notebook evaluates the **deployed** metadata-enrichment agent (`MetadataRuntimeArn`)
**entirely server-side** using Amazon Bedrock **AgentCore Batch Evaluations**
(`bedrock_agentcore.evaluation.BatchEvaluationRunner`).

We define a `Dataset` of scenarios, supply an `agent_invoker` that drives the deployed
runtime once per scenario (and **blocks until the async enrichment completes** so the
agent's spans exist), and `BatchEvaluationRunner` discovers the resulting CloudWatch
spans and runs the evaluators **in the AgentCore Evaluations service**.

### Evaluators (all server-side, AgentCore)
| Evaluator | What it checks |
|---|---|
| `Builtin.GoalSuccessRate` | Did the agent achieve the enrichment goal (assertions optional)? |
| `Builtin.ToolSelectionAccuracy` | Did it pick the right tools? |
| `Builtin.ToolParameterAccuracy` | Were the tool inputs reasonable? |

### Performance metrics (observability-derived)
Alongside the quality scores, the `agent_invoker` also captures cost/performance signals per
scenario — reported in the results table and persisted JSON, **separate** from the evaluator scores:

| Metric | Source |
|---|---|
| **Server-side latency** (`enrich_seconds`) | `completedAt − buildStartedAt` from the agent's DynamoDB record |
| **Token usage** (`input`/`output`/`total`) | `gen_ai.usage.*` summed over the session's CloudWatch OTel spans via `ObservabilityClient` |


## Prerequisites

- The **agentcore stack** is deployed (`{PROJECT_NAME}-agentcore`) and `MetadataRuntimeArn` is populated.
- `bedrock-agentcore>=1.11.0` (provides `bedrock_agentcore.evaluation.BatchEvaluationRunner`) installed in the kernel.
- AWS credentials (`AWS_PROFILE`) with access to:
  - **Bedrock AgentCore** — invoke runtime + StartBatchEvaluation + read results
  - **DynamoDB** — seed the metadata config + poll enrichment status
  - **Glue** — discover the namespace tables for the test case
  - **CloudWatch Logs** — the batch service discovers agent spans here


In [ ]:
# Verify the AgentCore Evaluation SDK is available
try:
    import bedrock_agentcore  # noqa: F401
    from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
        BatchEvaluationRunner,  # noqa: F401
    )
    print("✓ bedrock-agentcore installed:",
          getattr(bedrock_agentcore, "__version__", "version unknown"))
    print("✓ BatchEvaluationRunner import OK")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("  Install with: pip install 'bedrock-agentcore>=1.11.0'")
    raise


In [ ]:
import re

def _redact_account_ids(obj):
    """Recursively replace AWS account IDs embedded in ARN strings with XXXXXXXXXXXX."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(arn:[^:]*:[^:]*:[^:]*:)\d{12}:', r'\1XXXXXXXXXXXX:', obj)
    return obj


In [ ]:
# Imports — the deployed runtime is invoked over the network; evaluation runs in AgentCore.
import os
import sys  # noqa: F401  (kept for parity with sibling eval notebooks)
import json
import time
import uuid
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone

from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig,
    BatchEvaluatorConfig,
    CloudWatchDataSourceConfig,
)
from bedrock_agentcore.evaluation.runner.dataset_types import (
    Dataset,
    PredefinedScenario,
    Turn,
)
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput,
    AgentInvokerOutput,
)

print("✓ Imports successful")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


In [ ]:
# Load environment variables from .env file (sourced from cdk/cdk-outputs-agentcore.json)
from dotenv import load_dotenv, set_key

ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

# Create boto3 session early — needed for CloudFormation lookups below
config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'default'))
region = session.region_name or 'us-east-1'

# ── CloudFormation output resolver ──────────────────────────────────────────
# For any env var still set to 'XXX' (no .env or .env missing the key),
# resolve the value from live CloudFormation stack outputs.

def _get_stack_outputs(stack_name: str) -> dict:
    """Return {OutputKey: OutputValue} for a CloudFormation stack, or {} if not found."""
    cfn = session.client('cloudformation', region_name=region)
    try:
        resp = cfn.describe_stacks(StackName=stack_name)
        return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
    except Exception:
        return {}

# Map: env_var_name → (stack_name, output_key)
_CFN_MAP = {
    'ARTIFACTS_BUCKET':            (f'{PROJECT_NAME}-data-lake',  'ArtifactsBucketName'),
    'ATHENA_WORKGROUP':            (f'{PROJECT_NAME}-athena',     'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE':     (f'{PROJECT_NAME}-dynamodb',   'MetadataTableName'),
    'SEMANTIC_RAG_KB_ID':          (f'{PROJECT_NAME}-bedrock-kb', 'SemanticRagKbId'),
    'SEMANTIC_RAG_DATA_SOURCE_ID': (f'{PROJECT_NAME}-bedrock-kb', 'SemanticRagDataSourceId'),
    'KNOWLEDGE_BASE_ID':           (f'{PROJECT_NAME}-bedrock-kb', 'OntologyPatternsKbId'),
}

_cfn_cache = {}
_resolved_from_cfn = []

for env_var, (stack, output_key) in _CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        os.environ[env_var] = current
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)

if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

print("\n✓ Environment variables configured:")
for key in ['ARTIFACTS_BUCKET', 'ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE',
            'SEMANTIC_RAG_KB_ID', 'SEMANTIC_RAG_DATA_SOURCE_ID', 'KNOWLEDGE_BASE_ID']:
    source = '(from CloudFormation → .env)' if key in _resolved_from_cfn else '(from .env)'
    val = os.environ.get(key, '(not set)')
    print(f"  {key} = {val}  {source}")

In [ ]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

In [ ]:
# Verify credentials (session created in previous cell)
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified:")
print(f"  Profile:   {os.environ.get('AWS_PROFILE', 'default')}")
print(f"  Account:   ...{account_id[-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region:    {region}")

## Resolve the AgentCore Metadata Runtime

Read the deployed metadata-enrichment runtime ARN from the `{PROJECT_NAME}-agentcore` CloudFormation stack and derive the agent ID from it. The agent ID is what AgentCore Evaluations keys evaluator results on.

In [ ]:
cfn = session.client('cloudformation', region_name=region)
ac_outputs = cfn.describe_stacks(StackName=f'{PROJECT_NAME}-agentcore')['Stacks'][0]['Outputs']
ac_output_map = {o['OutputKey']: o['OutputValue'] for o in ac_outputs}
METADATA_RUNTIME_ARN = ac_output_map['MetadataRuntimeArn']
agent_id = METADATA_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ Runtime ARN: {METADATA_RUNTIME_ARN}")
print(f"✓ Agent ID:    {agent_id}")

In [ ]:
# # ── (Optional) Hardcode the semantic-layer id for downstream notebooks ──
# # This is where the layer id is shared across notebooks: notebook 1 sets the
# # `metadata_id` global and persists it with IPython's `%store`; notebooks 2/3/…
# # restore it with `%store -r metadata_id` (then validate it against the metadata
# # table, falling back to the latest completed semantic-rag layer if stale).
# #
# # Normally `metadata_id` comes from the Phase-1 enrichment run below, which takes
# # ~1 hour. To test the query/eval notebooks against an EXISTING completed layer
# # WITHOUT re-running Phase 1, set the id here, run this cell, and skip straight to
# # notebook 2. Leave the string empty to use the id produced by the Phase-1 run.
# # HARDCODE_METADATA_ID = "semantic-rag-multi_table_curated_layer-e80174d7"

# if HARDCODE_METADATA_ID:
#     metadata_id = HARDCODE_METADATA_ID
#     # `%store` magic must be its own statement (an inline comment after it is
#     # mis-parsed by IPython).
#     %store metadata_id
#     print(f"✓ Hardcoded metadata_id and stored for downstream notebooks: {metadata_id}")
# else:
#     print("• HARDCODE_METADATA_ID empty — metadata_id will come from the Phase-1 run.")


Stored 'metadata_id' (str)
✓ Hardcoded metadata_id and stored for downstream notebooks: semantic-rag-multi_table_curated_layer-e80174d7


## Define Test Cases — S3 Tables (Iceberg) Discovery

Discover all tables in the curated namespace and define one test case = one config = one runtime invocation = one session of N tables.

`MAX_TABLES` lets you smoke-test with 1 table; set to 0 for the full namespace.

In [ ]:
# S3 Tables (Iceberg) data source config
# CatalogId format: s3tablescatalog/<bucket-name>  (e.g. s3tablescatalog/semantic-layer-analytics-tables)
# Namespace: e.g. normalized

# Resolve catalog/database vars from CloudFormation if still 'XXX'
_CATALOG_CFN_MAP = {
    'S3T_DATABASE':      (f'{PROJECT_NAME}-data-lake',     'Namespace'),
    'DYNAMODB_CATALOG':  (f'{PROJECT_NAME}-athena',        'DynamoDBCatalogName'),
    'DYNAMODB_DATABASE': (f'{PROJECT_NAME}-glue-catalog',  'DynamoDBDatabaseName'),
}

for env_var, (stack, output_key) in _CATALOG_CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)

# S3T_CATALOG is special: s3tablescatalog/<bucket-name>
S3T_CATALOG = os.environ.get('S3T_CATALOG', 'XXX')
if S3T_CATALOG == 'XXX':
    if f'{PROJECT_NAME}-data-lake' not in _cfn_cache:
        _cfn_cache[f'{PROJECT_NAME}-data-lake'] = _get_stack_outputs(f'{PROJECT_NAME}-data-lake')
    prefix = _cfn_cache[f'{PROJECT_NAME}-data-lake'].get('S3TablesFederatedCatalogName', 's3tablescatalog')
    S3T_CATALOG = f"{prefix}/{PROJECT_NAME}-analytics-tables"
    os.environ['S3T_CATALOG'] = S3T_CATALOG
    _resolved_from_cfn.append('S3T_CATALOG')

S3T_DATABASE      = os.environ.get('S3T_DATABASE', 'XXX')
DYNAMODB_CATALOG  = os.environ.get('DYNAMODB_CATALOG', 'XXX')
DYNAMODB_DATABASE = os.environ.get('DYNAMODB_DATABASE', 'XXX')

_catalog_resolved = [k for k in ['S3T_CATALOG', 'S3T_DATABASE', 'DYNAMODB_CATALOG', 'DYNAMODB_DATABASE']
                     if k in _resolved_from_cfn]
if _catalog_resolved:
    for env_var in _catalog_resolved:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_catalog_resolved)} catalog variable(s) to .env: {', '.join(_catalog_resolved)}")

print("\n✓ Catalog/database configured:")
for key, val in [('S3T_CATALOG', S3T_CATALOG), ('S3T_DATABASE', S3T_DATABASE),
                 ('DYNAMODB_CATALOG', DYNAMODB_CATALOG), ('DYNAMODB_DATABASE', DYNAMODB_DATABASE)]:
    source = '(from CloudFormation → .env)' if key in _resolved_from_cfn else '(from .env)'
    print(f"  {key} = {val}  {source}")

def s3t(table_name: str) -> dict:
    """Build the S3 Tables (Iceberg) data-source dict the agent expects in DynamoDB."""
    return {
        'databaseName': S3T_DATABASE,
        'tableName':    table_name,
        'catalogId':    S3T_CATALOG,
        'dataSource':   'AwsDataCatalog',
        'tableId':      f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

# ── Discover all tables in the S3 Tables (curated/normalized) namespace ────
# Federated S3 Tables catalogs require CatalogId of "<account>:<S3T_CATALOG>"
glue_catalog_id = f"{account_id}:{S3T_CATALOG}"

glue = session.client('glue', region_name=region)
paginator = glue.get_paginator('get_tables')
all_table_names = []
for page in paginator.paginate(CatalogId=glue_catalog_id, DatabaseName=S3T_DATABASE):
    all_table_names.extend(t['Name'] for t in page.get('TableList', []))

all_table_names.sort()
print(f"\n✓ Discovered {len(all_table_names)} tables in {S3T_CATALOG}.{S3T_DATABASE}")

# Smoke-test slice: set MAX_TABLES=1 for a one-table run (~3-5 min); 0 = full namespace.
MAX_TABLES = int(os.environ.get('MAX_TABLES', '0'))
if MAX_TABLES > 0:
    selected_table_names = all_table_names[:MAX_TABLES]
    print(f"  ⚠ MAX_TABLES={MAX_TABLES} — using a {len(selected_table_names)}-table subset")
else:
    selected_table_names = list(all_table_names)
    print(f"  Using all {len(selected_table_names)} tables")

if not selected_table_names:
    raise RuntimeError(f"No tables found in {S3T_CATALOG}.{S3T_DATABASE} — cannot build test case")

test_cases = [
    {
        'id': 'multi_table_curated_layer',
        'category': 'multi_s3table',
        'dynamodb_config': {
            'name': 'curated-layer-s3tables',
            'type': 'SemanticRAG',
            'dataSourcesDescription': f'All {len(selected_table_names)} Iceberg S3 Tables in the {S3T_DATABASE} namespace',
            'useCasesDescription': 'Enable business users to query the entire curated semantic layer via Bedrock KB',
            'dataSources': [s3t(name) for name in selected_table_names],
        },
        'expected_assertion': (
            'For every table in the input dataSources, the agent should: '
            '(1) call get_single_table_schema, '
            '(2) call sample_table_data, '
            '(3) call update_glue_table_metadata with non-empty descriptions, '
            '(4) call save_metadata_document_to_s3, and '
            '(5) call update_progress. '
            'The final DynamoDB status must be "completed".'
        ),
    },
]

print(f"\n✓ Defined {len(test_cases)} test case(s)")
for tc in test_cases:
    n = len(tc['dynamodb_config']['dataSources'])
    print(f"  {tc['id']} ({tc['category']}): {n} table(s)")

In [ ]:
# helper method to create entry in semantic-metadata-layer table, replicating UI application logic

METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')

def seed_test_ontology(
    ontology_id: str,
    tables: list,
    name: str = 'semantic-rag-eval-test',
    data_sources_description: str = '',
    use_cases_description: str = '',
    created_by: str = 'eval@semantic-layer.local',
) -> None:
    """Write a DynamoDB v1 record matching the shape created by the admin UI flow.

    Parameters:
        ontology_id: the `id` partition key the agent will read.
        tables: list of full data-source dicts (from `s3t()`).
        name: human-readable config name; gets a UTC timestamp suffix.
        data_sources_description: optional free-text description of the sources.
        use_cases_description: optional free-text description of the intended use cases.
        created_by: user/email recorded as creator.
    """
    dynamodb = session.resource('dynamodb')
    ddb_table = dynamodb.Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    name_with_ts = f"{name}-{ts}"
    data_sources = [
        {
            'databaseName': t['databaseName'],
            'tableName':    t['tableName'],
            'catalogId':    t.get('catalogId', 'AWSDataCatalog'),
            'dataSource':   t.get('dataSource', 'AwsDataCatalog'),
            'tableId':      t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]

    item = {
        'id':             ontology_id,
        'version':        'v1',
        'type':           'SemanticRAG',
        'name':           name_with_ts,
        'status':         'pending',
        'configuration':  {},
        'dataSources':    data_sources,
        'createdAt':      now,
        'updatedAt':      now,
        'buildStartedAt': now,
        'createdBy':      created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description

    ddb_table.put_item(Item=item)
    print(f"✓ Seeded DynamoDB item: {ontology_id} → name='{name_with_ts}' ({len(data_sources)} table(s))")

print("✓ seed_test_ontology defined")

In [ ]:
# Observability helper — sum token usage from the agent's CloudWatch OTel spans.
#
# The enrichment agent is OTel-instrumented (Strands [otel]); each LLM cycle span carries
# gen_ai.usage.* attributes. We sum them across all spans in a session (the async background
# thread propagates the runtime session.id, so its spans are correlated to the same session).
# Reuses the pattern from 3_metadata_query_agent_ac_eval-online.ipynb.
from bedrock_agentcore_starter_toolkit.operations.observability import ObservabilityClient

obs_client = ObservabilityClient(region_name=region)


def _sum_session_tokens(*, session_id: str, start_ms: int, end_ms: int) -> dict:
    """Sum gen_ai.usage.* token attributes across every span in a session.

    Parameters:
        session_id: the runtime session id whose spans to aggregate.
        start_ms: query window start (epoch milliseconds).
        end_ms: query window end (epoch milliseconds).

    Returns:
        dict with integer 'inputTokens', 'outputTokens', 'totalTokens'.
    """
    spans = obs_client.query_spans_by_session(
        session_id=session_id,
        start_time_ms=start_ms,
        end_time_ms=end_ms,
        agent_id=agent_id,
    )
    in_t = out_t = tot_t = 0
    for span in spans:
        attrs = getattr(span, "attributes", {}) or {}
        in_t += int(attrs.get("gen_ai.usage.input_tokens", 0) or 0)
        out_t += int(attrs.get("gen_ai.usage.output_tokens", 0) or 0)
        tot_t += int(attrs.get("gen_ai.usage.total_tokens", 0) or 0)
    return {"inputTokens": in_t, "outputTokens": out_t, "totalTokens": tot_t}


print("✓ ObservabilityClient ready — _sum_session_tokens defined")


## Build the Evaluation Dataset + agent_invoker

`BatchEvaluationRunner.run_dataset_evaluation(config, dataset, agent_invoker)` invokes the
agent **once per scenario** via our `agent_invoker`, waits for CloudWatch ingestion, then
submits a single batch-evaluation job and polls until completion.

**The enrichment agent is asynchronous** (background thread + DynamoDB progress), so the
invoker must:
1. seed a fresh DynamoDB config (`id`) for this scenario,
2. `invoke_agent_runtime` with the framework-managed `session_id` as `runtimeSessionId`
   (so the batch service can correlate spans),
3. **block** by polling DynamoDB until status is `completed`/`failed`,
4. return the agent's summary as `agent_output`.

`Turn.input` / `PredefinedScenario.assertions` / `expected_trajectory` are the ground-truth
fields the built-in evaluators consume.


In [ ]:
agentcore_client = session.client("bedrock-agentcore", region_name=region)
ddb_resource = session.resource("dynamodb")
ddb_table = ddb_resource.Table(METADATA_TABLE)

# Map each scenario_id -> the seeded DynamoDB config id + table list, so the
# results cell can report which enrichment run each session corresponds to.
SCENARIO_RUNS: dict = {}


def _server_side_seconds(*, item: dict) -> float | None:
    """Compute enrichment latency from the DynamoDB record's own timestamps.

    The agent stamps `buildStartedAt` on the processing transition and `completedAt`
    on completion (see agents/metadata_agent/main.py). This is the true server-side
    work time, excluding invoke/poll overhead. Returns None if either timestamp is
    missing (e.g. a failed run) — we never fabricate a value.

    Parameters:
        item: the final DynamoDB item for the enrichment job.

    Returns:
        elapsed seconds as float, or None if timestamps are unavailable/unparseable.
    """
    started_raw = item.get("buildStartedAt")
    completed_raw = item.get("completedAt")
    if not started_raw or not completed_raw:
        return None
    try:
        started = datetime.fromisoformat(started_raw)
        completed = datetime.fromisoformat(completed_raw)
    except (TypeError, ValueError):
        return None
    return (completed - started).total_seconds()


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the deployed metadata-enrichment agent for one scenario and block
    until the async enrichment finishes (so its spans exist for batch discovery).

    BatchEvaluationRunner supplies:
      - invoker_input.payload     -> the Turn.input (the scenario id string here)
      - invoker_input.session_id  -> framework-managed runtimeSessionId
    """
    scenario_id = (
        invoker_input.payload if isinstance(invoker_input.payload, str)
        else json.dumps(invoker_input.payload)
    )
    tc = _TC_BY_SCENARIO[scenario_id]
    cfg = tc["dynamodb_config"]
    tables = cfg["dataSources"]
    num_tables = len(tables)

    case_id = f"semantic-rag-{tc['id']}-{uuid.uuid4().hex[:8]}"
    print(f"\n=== scenario '{scenario_id}' -> case_id={case_id} ({num_tables} tables) ===")
    print(f"  runtimeSessionId = {invoker_input.session_id}")

    seed_test_ontology(
        case_id, tables,
        name=cfg["name"],
        data_sources_description=cfg.get("dataSourcesDescription", ""),
        use_cases_description=cfg.get("useCasesDescription", ""),
    )

    # Bracket the token-usage query window: from just before invoke to just after
    # completion (padded). Tighter than a fixed lookback — avoids cross-scenario span bleed.
    invoke_start_ms = int(time.time() * 1000)

    _invoke_runtime(
        METADATA_RUNTIME_ARN,
        invoker_input.session_id,
        json.dumps({"id": case_id}).encode("utf-8"),
    )

    # Block until the background enrichment thread reports completion.
    # ~60s/table observed; budget 90s/table with a 5-min floor.
    max_wait_secs = max(300, 90 * num_tables)
    start = datetime.now()
    final_item, final_status = {}, "processing"
    print(f"  Polling DynamoDB for completion (max {max_wait_secs}s)...")
    while (datetime.now() - start).total_seconds() < max_wait_secs:
        time.sleep(30)
        final_item = ddb_table.get_item(Key={"id": case_id, "version": "v1"}).get("Item", {})
        final_status = final_item.get("status", "unknown")
        progress = final_item.get("progress", {}) or {}
        pm = f" — {progress.get('processed', '?')}/{progress.get('total', '?')}" if progress else ""
        print(f"    [{(datetime.now()-start).total_seconds():.0f}s] status={final_status}{pm}")
        if final_status in ("completed", "failed"):
            break

    summary = final_item.get("summary") or final_item.get("error", "") or f"status={final_status}"

    # ── Latency (server-side): completedAt − buildStartedAt from the DDB record ──
    enrich_seconds = _server_side_seconds(item=final_item)

    # ── Token usage: sum gen_ai.usage.* across this session's CloudWatch spans ──
    # Best-effort — observability ingestion can lag; never fail the eval over it.
    invoke_end_ms = int(time.time() * 1000) + 5000  # small pad for the last span to land
    usage = {"inputTokens": None, "outputTokens": None, "totalTokens": None}
    try:
        usage = _sum_session_tokens(
            session_id=invoker_input.session_id,
            start_ms=invoke_start_ms,
            end_ms=invoke_end_ms,
        )
    except Exception as exc:  # noqa: BLE001 — observability fetch is best-effort
        print(f"  ⚠ token-usage fetch failed: {exc}")

    SCENARIO_RUNS[scenario_id] = {
        "case_id": case_id,
        "session_id": invoker_input.session_id,
        "status": final_status,
        "num_tables": num_tables,
        "summary": summary,
        "enrich_seconds": round(enrich_seconds, 1) if enrich_seconds is not None else None,
        "input_tokens": usage["inputTokens"],
        "output_tokens": usage["outputTokens"],
        "total_tokens": usage["totalTokens"],
        # CloudWatch query window — reused by the hallucination-check cell to scope
        # the guard's "Schema validation dropped ..." WARNING scan to THIS run.
        "invoke_start_ms": invoke_start_ms,
        "invoke_end_ms": invoke_end_ms,
    }
    # Stash the most recent metadata id so downstream notebooks can reuse it.
    global metadata_id
    metadata_id = case_id
    lat_str = f"{enrich_seconds:.1f}s" if enrich_seconds is not None else "n/a"
    print(f"  {'✓' if final_status == 'completed' else '✗'} {final_status} — {summary}")
    print(f"     latency(server)={lat_str}  tokens(in/out/total)="
          f"{usage['inputTokens']}/{usage['outputTokens']}/{usage['totalTokens']}")
    return AgentInvokerOutput(agent_output=summary)


# One scenario per test case. assertions / expected_trajectory are the ground
# truth the SESSION-level built-in evaluators score against.
_TC_BY_SCENARIO = {tc["id"]: tc for tc in test_cases}

dataset = Dataset(
    scenarios=[
        PredefinedScenario(
            scenario_id=tc["id"],
            turns=[Turn(input=tc["id"])],  # payload routes the invoker to this tc
            expected_trajectory=[
                "get_single_table_schema",
                "sample_table_data",
                "update_glue_table_metadata",
                "save_metadata_document_to_s3",
                "update_progress",
            ],
            assertions=[
                "The agent enriched metadata for every table in the input dataSources.",
                "update_glue_table_metadata was called with non-empty column descriptions.",
                "save_metadata_document_to_s3 persisted a markdown doc per table.",
                "The final DynamoDB status is 'completed'.",
            ],
        )
        for tc in test_cases
    ]
)
print(f"✓ Dataset built: {len(dataset.scenarios)} scenario(s)")
for s in dataset.scenarios:
    print(f"  {s.scenario_id}: {len(s.assertions)} assertion(s), "
          f"{len(s.expected_trajectory)} expected tool(s)")


## Phase 1 — Invoke the agent (long-running) and **persist** the sessions

`BatchEvaluationRunner.run_dataset_evaluation` couples three things into one blocking
call: it (1) invokes the agent per scenario, (2) waits for span ingestion, and (3) submits
+ polls the batch job. For the metadata-enrichment agent, step (1) is a **single ~1-hour
runtime invocation** — so if the batch poll is interrupted (Ctrl-C, timeout, transient
error), the entire hour of enrichment is thrown away and `batch_result` never binds (which
is exactly what happened on the prior run).

This notebook **decouples** the two phases:

- **Phase 1 (this cell)** runs the invocation exactly as the SDK's executor would — same
  `session_id = "{scenario_id}-{uuid4}"` format, same per-turn loop — then captures each
  session's `(session_id, start_time, end_time, ground_truth)` and **writes them to a
  recovery file on disk**. The expensive work is saved the instant it finishes.
- **Phase 2 (next cell)** submits the batch *against those already-completed sessions* and
  polls. It is **idempotent / re-runnable**: an interrupted or `FAILED` poll costs seconds,
  not an hour, and it reloads the sessions from disk after a kernel restart.

Ground-truth transformation, the CloudWatch data-source config, and the poll loop all reuse
the SDK's own methods (`_transform_ground_truth`, `to_data_source_config`,
`_poll_for_results`) so the decoupled path stays bit-for-bit consistent with the library.


In [ ]:
import uuid as _uuid
from datetime import datetime, timezone

# Span-discovery coordinates (also reused by Phase 2 below and the grounding cell).
SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

# Recovery file: Phase 1 writes the completed sessions here; Phase 2 reads them. Survives a
# kernel restart, so the ~1-hour enrichment never has to be re-run to (re)submit the batch.
SESSIONS_FILE = "../data/eval/results/metadata_sessions_latest.json"
os.makedirs("../data/eval/results", exist_ok=True)

batch_runner = BatchEvaluationRunner(region=region)

# ── Phase 1: run each scenario's invocation ourselves ────────────────────────
# This mirrors PredefinedScenarioExecutor.run_scenario (scenario_executor.py): generate the
# same session_id format, invoke agent_invoker once per turn, and bracket the run with UTC
# start/end timestamps. We do it inline (not via run_dataset_evaluation) so the result is
# captured + persisted BEFORE any batch submission can fail.
collected_sessions = []   # list of dicts: sessionId / testScenarioId / window / groundTruth
for scenario in dataset.scenarios:
    sid = f"{scenario.scenario_id}-{_uuid.uuid4()}"
    start_time = datetime.now(timezone.utc)
    print(f"\n▶ Phase 1: scenario '{scenario.scenario_id}' (session_id={sid})")
    for turn in scenario.turns:
        agent_invoker(AgentInvokerInput(payload=turn.input, session_id=sid))
    end_time = datetime.now(timezone.utc)

    # Reuse the SDK's own ground-truth transform so assertions / expected_trajectory are
    # encoded identically to run_dataset_evaluation (no drift).
    ground_truth = batch_runner._transform_ground_truth(scenario)
    collected_sessions.append({
        "sessionId": sid,
        "testScenarioId": scenario.scenario_id,
        "startTime": start_time.isoformat(),
        "endTime": end_time.isoformat(),
        "groundTruth": ground_truth,
    })
    print(f"  ✓ session captured: window=[{start_time.isoformat()}, {end_time.isoformat()}]")

# ── Persist the sessions immediately — this is the recovery point ────────────
# SCENARIO_RUNS (latency/tokens/invoke-window) is also persisted so the grounding cell and
# the results cell can run after a restart without re-invoking the agent.
with open(SESSIONS_FILE, "w") as f:
    json.dump(
        {
            "agent_id": agent_id,
            "runtime_arn": METADATA_RUNTIME_ARN,
            "service_name": SERVICE_NAME,
            "log_groups": [SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
            "sessions": collected_sessions,
            "scenario_runs": SCENARIO_RUNS,
        },
        f, indent=2, default=str,
    )

print(f"\n✓ Phase 1 complete — {len(collected_sessions)} session(s) persisted to {SESSIONS_FILE}")
print("  The enrichment is now saved. Phase 2 (next cell) can submit/re-submit the batch")
print("  cheaply — even after a kernel restart — without re-invoking the agent.")


## Phase 2 — Submit + poll the batch (idempotent, re-runnable)

Submits a single `StartBatchEvaluation` job against the sessions captured in Phase 1 and
polls to completion. **This cell is safe to re-run on its own** — it loads the sessions from
`SESSIONS_FILE` if they aren't in memory (e.g. after a kernel restart), so a Ctrl-C, a poll
timeout, or a `FAILED` status never forces a re-invocation of the ~1-hour agent.

The submission + poll logic mirrors `run_dataset_evaluation`'s second half exactly — same
`pre_evaluation_run_hook` (span-ingestion wait), same `start_batch_evaluation` arguments,
same `_poll_for_results` loop, same `BatchEvaluationResult` shape — but reads its sessions
from disk instead of from a just-completed in-memory invocation.

> A non-successful terminal status (`FAILED`) still binds `batch_result` with whatever
> `evaluationResults` the service returned, so the downstream results cell can persist
> partial scores rather than crashing (the prior failure mode).


In [ ]:
from datetime import datetime
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationResult,
    BatchEvaluationSummary,
    CloudWatchOutputDataConfig,
)

# Re-runnable: prefer in-memory sessions from Phase 1, else reload from disk (kernel restart).
if "collected_sessions" not in dir() or not collected_sessions:
    with open(SESSIONS_FILE) as f:
        _saved = json.load(f)
    collected_sessions = _saved["sessions"]
    # Rehydrate the coordinates Phase 2 + downstream cells need.
    agent_id = _saved["agent_id"]
    METADATA_RUNTIME_ARN = _saved["runtime_arn"]
    SERVICE_NAME = _saved["service_name"]
    SPANS_LOG_GROUP, RUNTIME_LOG_GROUP = _saved["log_groups"]
    SCENARIO_RUNS = _saved.get("scenario_runs", {})
    print(f"✓ Reloaded {len(collected_sessions)} session(s) from {SESSIONS_FILE} (Phase 1 not in memory)")

if "batch_runner" not in dir():
    batch_runner = BatchEvaluationRunner(region=region)

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=180,  # enrichment spans land seconds before the DDB flip
)

EVALUATOR_IDS = [
    "Builtin.GoalSuccessRate",         # SESSION
    "Builtin.ToolSelectionAccuracy",   # tool
    "Builtin.ToolParameterAccuracy",   # tool
]
POLLING_TIMEOUT_SECONDS = 5400
POLLING_INTERVAL_SECONDS = 30

# ── Parse the persisted session windows back into datetimes ──────────────────
_session_ids = [s["sessionId"] for s in collected_sessions]
_starts = [datetime.fromisoformat(s["startTime"]) for s in collected_sessions]
_ends = [datetime.fromisoformat(s["endTime"]) for s in collected_sessions]
_session_metadata = [
    {
        "sessionId": s["sessionId"],
        "testScenarioId": s["testScenarioId"],
        **({"groundTruth": {"inline": s["groundTruth"]}} if s.get("groundTruth") else {}),
    }
    for s in collected_sessions
]

batch_name = f"metadata_enrich_batch_{uuid.uuid4().hex[:8]}"
print(f"Batch name : {batch_name}")
print(f"Evaluators : {EVALUATOR_IDS}")
print(f"Sessions   : {len(_session_ids)}")
print("Waiting for span ingestion, then submitting StartBatchEvaluation...\n")

# 1) Wait for CloudWatch span ingestion (same hook run_dataset_evaluation calls).
batch_data_source.pre_evaluation_run_hook()

# 2) Submit the batch against the EXISTING sessions (no agent invocation here).
_start_kwargs = dict(
    batchEvaluationName=batch_name,
    evaluators=[{"evaluatorId": eid} for eid in EVALUATOR_IDS],
    dataSourceConfig=batch_data_source.to_data_source_config(
        _session_ids, min(_starts), max(_ends),
    ),
    evaluationMetadata={"sessionMetadata": _session_metadata},
    description="Built-in span-based evaluation of the deployed metadata-enrichment agent.",
)
_start_response = batch_runner.data_plane_client.start_batch_evaluation(**_start_kwargs)
_batch_id = _start_response["batchEvaluationId"]
_batch_arn = _start_response["batchEvaluationArn"]
print(f"✓ Started batch: {_batch_id}")

# 3) Poll to a terminal state via the SDK's own poll loop.
_response = batch_runner._poll_for_results(
    _batch_id, POLLING_TIMEOUT_SECONDS, POLLING_INTERVAL_SECONDS,
)

# 4) Assemble a BatchEvaluationResult identically to run_dataset_evaluation — a FAILED
#    terminal status still binds batch_result with whatever evaluationResults exist.
_eval_results = (
    BatchEvaluationSummary.model_validate(_response["evaluationResults"])
    if "evaluationResults" in _response else None
)
_output_data_config = None
if "outputConfig" in _response:
    _odc = _response["outputConfig"].get("cloudWatchConfig")
    if _odc:
        _output_data_config = CloudWatchOutputDataConfig(
            log_group_name=_odc["logGroupName"],
            log_stream_name=_odc["logStreamName"],
        )

batch_result = BatchEvaluationResult(
    batch_evaluation_id=_batch_id,
    batch_evaluation_arn=_batch_arn,
    batch_evaluation_name=_response["batchEvaluationName"],
    status=_response["status"],
    created_at=_response["createdAt"],
    description=_response.get("description"),
    evaluation_results=_eval_results,
    error_details=_response.get("errorDetails"),
    output_data_config=_output_data_config,
)
print(f"\n✓ Batch status: {batch_result.status}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")


## Schema-Hallucination Check (column / join grounding)

The three built-in evaluators above score **trajectory + goal** ("did it run the right
tools and finish?") — they do **not** verify that the `## Columns` / `## Reference Tables`
sections of each saved doc actually match the real table schema. A doc that invents
`holding.party_id` or a join on a non-existent `holding_subaccount.invest_product_id`
passes every built-in evaluator, yet poisons the query agent's Phase-3 slice and produces
ungroundable SQL.

This section adds the missing **content-grounding** signal in two complementary ways
(both reuse `agents/metadata_agent/doc_validator.validate_and_clean` — the same logic the
deployed agent's save-time guard uses, so the eval and the guard can never drift). It runs
**before** the persist cell so the results JSON captures the grounding numbers:

1. **Guard drops (is the model still fabricating?)** — scan the runtime log group for the
   guard's `Schema validation dropped …` WARNING within each scenario's invoke window.
   A non-zero count means the LLM *did* hallucinate but the deployed guard caught and
   stripped it before save. This is the true model-quality signal now that the guard
   rewrites docs before they land.
2. **Leaked hallucinations (did anything slip past the guard?)** — download every saved
   `.md` from S3, re-fetch the live schema, and re-run `validate_and_clean`. The expected
   result is **0** (the guard already cleaned them); a non-zero count is a guard
   regression or a pre-guard doc lingering in S3 — asserted to be zero.


In [ ]:
# Schema-hallucination check — reuses the deployed agent's own validator so the
# eval signal and the save-time guard can never drift apart.
import importlib.util as _ilu

# Import the pure validator directly by file path (no package import side effects;
# doc_validator depends only on stdlib re/typing).
_dv_path = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd(),
    "..", "agents", "metadata_agent", "doc_validator.py",
)
_spec = _ilu.spec_from_file_location("doc_validator", _dv_path)
doc_validator = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(doc_validator)
print(f"✓ Loaded validator from {os.path.relpath(_dv_path)}")

logs_client = session.client("logs", region_name=region)
s3_client = session.client("s3", region_name=region)
ARTIFACTS_BUCKET = os.environ["ARTIFACTS_BUCKET"]


def _real_columns(database_name: str, table_name: str) -> set:
    """Lower-cased real column names for a curated S3 Tables (Iceberg) table.

    Mirrors what the deployed agent's _fetch_real_columns resolves, but read here
    via Glue (the federated S3 Tables catalog) so the notebook needs no runtime call.
    Returns an empty set when the table can't be resolved (→ validation skipped, as
    in the agent) so a transient Glue miss never produces a false 'leak'.
    """
    try:
        tbl = glue.get_table(
            CatalogId=f"{account_id}:{S3T_CATALOG}",
            DatabaseName=database_name,
            Name=table_name,
        )["Table"]
    except Exception as exc:  # noqa: BLE001
        print(f"    ⚠ could not resolve schema for {database_name}.{table_name}: {exc}")
        return set()
    sd = tbl.get("StorageDescriptor", {})
    cols = {c["Name"].lower() for c in sd.get("Columns", [])}
    cols |= {pk["Name"].lower() for pk in tbl.get("PartitionKeys", [])}
    return cols


def _count_guard_drops(*, start_ms: int, end_ms: int) -> tuple:
    """Count the guard's 'Schema validation dropped ...' WARNINGs in the runtime
    log group within a time window. Returns (event_count, dropped_identifier_count).

    Each WARNING line reports one table's drop event and lists the dropped ids, e.g.
    '... dropped 3 hallucinated identifier(s) from normalized.holding before KB save:
    [column:party_id, ...]'. We sum both the events (tables affected) and the
    identifier counts (total fabrications stripped).
    """
    events = 0
    ids = 0
    try:
        paginator = logs_client.get_paginator("filter_log_events")
        for page in paginator.paginate(
            logGroupName=RUNTIME_LOG_GROUP,
            startTime=start_ms,
            endTime=end_ms,
            filterPattern='"Schema validation dropped"',
        ):
            for ev in page.get("events", []):
                events += 1
                m = re.search(r"dropped (\d+) hallucinated", ev["message"])
                if m:
                    ids += int(m.group(1))
    except logs_client.exceptions.ResourceNotFoundException:
        print(f"    ⚠ log group {RUNTIME_LOG_GROUP} not found")
    return events, ids


def _leaked_in_saved_docs(*, case_id: str) -> tuple:
    """Re-validate every saved .md doc for a run against the LIVE schema.

    Lists s3://{ARTIFACTS_BUCKET}/metadata/{case_id}/v1/... , downloads each .md,
    re-runs validate_and_clean, and returns (docs_checked, total_leaked_identifiers,
    sample_leaks). Expected to be 0 — the deployed guard already cleaned each doc
    before save, so any leak is a guard regression or a pre-guard doc lingering.
    """
    prefix = f"metadata/{case_id}/v1/"
    checked = 0
    leaked = 0
    samples = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=ARTIFACTS_BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(".md"):  # skip the .md.metadata.json sidecars
                continue
            # Key layout: metadata/{id}/v1/{catalog_id}/{database}/{table}.md
            parts = key[len(prefix):].split("/")
            if len(parts) < 3:
                continue
            database_name = parts[-2]
            table_name = parts[-1][: -len(".md")]
            md = s3_client.get_object(Bucket=ARTIFACTS_BUCKET, Key=key)["Body"].read().decode("utf-8")
            real_cols = _real_columns(database_name, table_name)
            if not real_cols:
                continue  # unresolved schema → skip (matches guard behaviour)
            target_cols = {
                edge["to"].lower(): _real_columns(database_name, edge["to"])
                for edge in doc_validator.extract_reference_edges(md)
                if edge.get("to")
            }
            target_cols = {t: c for t, c in target_cols.items() if c}
            _, dropped = doc_validator.validate_and_clean(
                md=md, real_columns=real_cols, target_columns=target_cols,
            )
            checked += 1
            if dropped:
                leaked += len(dropped)
                samples.append({"table": f"{database_name}.{table_name}", "leaked": dropped})
    return checked, leaked, samples


# ── Run both checks per scenario ─────────────────────────────────────────────
halluc_rows = []
for scenario_id, run in SCENARIO_RUNS.items():
    case_id = run["case_id"]
    print(f"\n── {scenario_id} (case_id={case_id}) ──")

    drop_events, drop_ids = _count_guard_drops(
        start_ms=run.get("invoke_start_ms", 0),
        end_ms=run.get("invoke_end_ms", int(time.time() * 1000)),
    )
    print(f"  Guard drops      : {drop_ids} identifier(s) across {drop_events} table(s) "
          f"(model fabricated, guard caught)")

    docs_checked, leaked, leak_samples = _leaked_in_saved_docs(case_id=case_id)
    print(f"  Saved-doc re-val : {docs_checked} doc(s) checked, {leaked} leaked "
          f"(expected 0)")
    if leak_samples:
        for s in leak_samples[:5]:
            print(f"    ✗ LEAK {s['table']}: {s['leaked']}")

    run["guard_drop_events"] = drop_events
    run["guard_drop_identifiers"] = drop_ids
    run["docs_checked"] = docs_checked
    run["leaked_identifiers"] = leaked
    halluc_rows.append({
        "Scenario": scenario_id,
        "GuardDrops(ids)": drop_ids,
        "GuardDrops(tables)": drop_events,
        "DocsChecked": docs_checked,
        "Leaked": leaked,
    })

df_halluc = pd.DataFrame(halluc_rows)
display(df_halluc)

# Defense-in-depth assertion: the guard must leave ZERO hallucinations in saved docs.
_total_leaked = sum(r["leaked_identifiers"] for r in SCENARIO_RUNS.values())
if _total_leaked == 0:
    print(f"\n✓ Grounding check PASSED — 0 hallucinated identifiers leaked into saved docs.")
else:
    print(f"\n✗ Grounding check FAILED — {_total_leaked} hallucinated identifier(s) "
          f"leaked past the guard. Investigate doc_validator / re-run enrichment.")
assert _total_leaked == 0, (
    f"{_total_leaked} hallucinated identifier(s) leaked into saved KB docs — "
    f"the save-time guard in metadata_agent should have stripped them."
)


## Batch Results + Persist

Per-evaluator aggregate scores come straight from the AgentCore batch job
(`evaluation_results.evaluator_summaries`). We also persist the run metadata + the
per-scenario enrichment outcomes recorded by `agent_invoker`.


In [ ]:
os.makedirs("../data/eval/results", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Batch ID  : {batch_result.batch_evaluation_id}")
print(f"ARN       : {batch_result.batch_evaluation_arn}")
print(f"Status    : {batch_result.status}")

rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        rows.append({
            "Evaluator": es.evaluator_id or "unknown",
            "AvgScore": avg,
            "Evaluated": es.total_evaluated or 0,
            "Failed": es.total_failed or 0,
        })
else:
    print("\n⚠ No evaluation_results on the batch result — check job status / spans.")

df_batch = pd.DataFrame(rows)

# ---- aggregate performance metrics (latency + tokens) across scenarios -------
# Server-side latency (completedAt − buildStartedAt) and gen_ai.usage.* token sums
# come from the agent_invoker via SCENARIO_RUNS. These are observability-derived
# cost/performance signals, separate from the server-side evaluator scores above.
_lat_vals = [r["enrich_seconds"] for r in SCENARIO_RUNS.values() if r.get("enrich_seconds") is not None]
_in_total = sum(int(r.get("input_tokens") or 0) for r in SCENARIO_RUNS.values())
_out_total = sum(int(r.get("output_tokens") or 0) for r in SCENARIO_RUNS.values())
_tok_total = sum(int(r.get("total_tokens") or 0) for r in SCENARIO_RUNS.values())

performance = {
    "scenarios": len(SCENARIO_RUNS),
    "avg_enrich_seconds": round(sum(_lat_vals) / len(_lat_vals), 1) if _lat_vals else None,
    "total_enrich_seconds": round(sum(_lat_vals), 1) if _lat_vals else None,
    "input_tokens": _in_total,
    "output_tokens": _out_total,
    "total_tokens": _tok_total,
}

print("\n── Performance (latency + tokens) ──")
print(f"  Avg server-side latency : "
      f"{performance['avg_enrich_seconds']}s  (n={len(_lat_vals)})")
print(f"  Total tokens            : {_tok_total} (in={_in_total}, out={_out_total})")

# ---- schema-hallucination grounding signal (from the check cell below) -------
# Populated by the "Schema-Hallucination Check" cell; default to empty so this
# cell still persists cleanly if that cell hasn't run yet.
grounding = {
    "guard_drop_identifiers": sum(int(r.get("guard_drop_identifiers") or 0) for r in SCENARIO_RUNS.values()),
    "guard_drop_tables": sum(int(r.get("guard_drop_events") or 0) for r in SCENARIO_RUNS.values()),
    "docs_checked": sum(int(r.get("docs_checked") or 0) for r in SCENARIO_RUNS.values()),
    "leaked_identifiers": sum(int(r.get("leaked_identifiers") or 0) for r in SCENARIO_RUNS.values()),
}
print("\n── Schema grounding ──")
print(f"  Guard drops : {grounding['guard_drop_identifiers']} id(s) "
      f"across {grounding['guard_drop_tables']} table(s)")
print(f"  Leaked      : {grounding['leaked_identifiers']} "
      f"(of {grounding['docs_checked']} doc(s) re-validated; expected 0)")

# ---- persist combined results -------------------------------------------
combined_file = f"../data/eval/results/metadata_gen_batch_eval_{timestamp}.json"
combined = {
    "agent_id": agent_id,
    "runtime_arn": METADATA_RUNTIME_ARN,
    "batch_evaluation_id": batch_result.batch_evaluation_id,
    "batch_evaluation_arn": batch_result.batch_evaluation_arn,
    "status": str(batch_result.status),
    "evaluator_summaries": rows,
    "performance": performance,
    "grounding": grounding,
    "scenario_runs": SCENARIO_RUNS,
}
combined = _redact_account_ids(combined)
with open(combined_file, "w") as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

display(df_batch)
display(pd.DataFrame(SCENARIO_RUNS).T)


## Summary

This notebook evaluates the **deployed** metadata-enrichment agent **entirely inside
AgentCore Batch Evaluations** — no local scoring — plus a deterministic
**schema-grounding** check on the docs it produced.

### Pipeline
1. **Dataset** — one `PredefinedScenario` per test case, with `assertions` + an
   `expected_trajectory` (the ground truth the built-in evaluators score against).
2. **agent_invoker** — for each scenario: seed a DynamoDB config, `invoke_agent_runtime`
   with the framework session id, and **block** on DynamoDB until enrichment completes so
   the agent's CloudWatch spans exist. It also records **server-side latency**
   (`completedAt − buildStartedAt`), **token usage** (`gen_ai.usage.*` summed over the
   session's spans), and the **CloudWatch invoke window** (reused by the grounding check).
3. **Phase 1 — invoke + persist** — runs each scenario's invocation inline (same
   `session_id` format and per-turn loop as the SDK's `PredefinedScenarioExecutor`), then
   writes the completed sessions (`session_id` + time window + transformed ground truth) to
   `metadata_sessions_latest.json`. The ~1-hour enrichment is saved the instant it finishes.
4. **Phase 2 — submit + poll (idempotent)** — submits one `StartBatchEvaluation` job against
   those already-completed sessions (reloading from disk if the kernel restarted) and polls
   via the SDK's own `_poll_for_results`. Re-runnable: an interrupted/`FAILED` poll costs
   seconds, never a re-invocation. A `FAILED` status still binds `batch_result` with any
   partial `evaluationResults`.
5. **Schema-Hallucination Check** — runs before persist; see below.

> **Why two phases?** `BatchEvaluationRunner.run_dataset_evaluation` couples invoke + submit
> + poll into one blocking call. For this agent the invoke is a single ~1-hour runtime call,
> so a Ctrl-C or poll timeout used to discard the entire hour and leave `batch_result`
> unbound (the prior run's failure mode). Splitting the phases makes the expensive
> enrichment recoverable while reusing the SDK's own transform/config/poll methods so the
> decoupled path can't drift from the library.

### Evaluators (server-side)
- `Builtin.GoalSuccessRate` — goal achieved (uses the scenario assertions)
- `Builtin.ToolSelectionAccuracy` — right tools chosen
- `Builtin.ToolParameterAccuracy` — tool inputs reasonable

> **Why a separate grounding check?** The three built-ins score *trajectory + goal* — they
> do not verify that a saved doc's `## Columns` / `## Reference Tables` match the real
> table schema. A doc inventing `holding.party_id` or a join on a non-existent
> `holding_subaccount.invest_product_id` passes every built-in yet poisons the query
> agent's Phase-3 slice. The check below closes that gap deterministically.

### Schema-grounding signals (deterministic, reuse `doc_validator.validate_and_clean`)
- **Guard drops** — count of the deployed agent's `Schema validation dropped …` WARNINGs in
  the runtime log group per scenario. Non-zero = the LLM fabricated columns/joins and the
  save-time guard stripped them. This is the model-quality signal (the saved docs are
  already clean, so the model's raw error rate is only visible here).
- **Leaked hallucinations** — re-validate every saved `.md` against the live schema; any
  non-zero count is a guard regression or a pre-guard doc lingering in S3. **Asserted == 0.**

### Performance metrics (observability-derived, separate from evaluator scores)
- **Latency** — `enrich_seconds` per scenario, plus an avg/total aggregate.
- **Tokens** — `input`/`output`/`total` per scenario summed from `gen_ai.usage.*`, plus a
  run-level total.

### Output artifacts
- `../data/eval/results/metadata_sessions_latest.json` — Phase 1 recovery file: completed
  `session_id`s + time windows + transformed ground truth (+ `scenario_runs`). Lets Phase 2
  and the downstream cells run after a kernel restart without re-invoking the agent.
- `../data/eval/results/metadata_gen_batch_eval_{timestamp}.json` — batch id/arn, per-evaluator
  aggregate scores, a `performance` block, a `grounding` block (guard-drop + leaked counts),
  and per-scenario enrichment outcomes.

### Tuning knobs
- `MAX_TABLES` env var — slice the namespace for smoke tests (e.g. `MAX_TABLES=1`).
- `ingestion_delay_seconds` (180) — wait for CloudWatch span ingestion before evaluating.
- `POLLING_TIMEOUT_SECONDS` (5400) — covers the batch job; Phase 1 (enrichment) is no longer
  inside the polled window, so a timeout here no longer discards the enrichment.
